# Dependencies
Install necessary dependencies:
- tokenizers, transformers for tokenization
- torch for training

In [ ]:
%pip install tokenizers==0.21.1 transformers==4.51.3 torch==2.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
import torch
torch.cuda.device_count()

0

Let's first mount our Google Drive so that we can read the training data from it.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
MINI_SOURCE_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh-mini.en"
MINI_TARGET_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh-mini.zh"
SOURCE_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh.en"
TARGET_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh.zh"

# Tokenizer
First, we will train two tokenizers, one for the Chinese text and another one for the English text. Usually we should save the trained tokenizers but since the data size is small we can just train a new one in runtime.

In [5]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

special_tokens = ["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]", "[SOS]", "[EOS]"]

# Train one tokenizer for the English text
raw_en_tokenizer = Tokenizer(BPE())
raw_en_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=special_tokens)
raw_en_tokenizer.train(
  files=[SOURCE_FILE],
  trainer=trainer
)

# Train another tokenizer for the Chinese text
raw_zh_tokenizer = Tokenizer(BPE())
trainer = BpeTrainer(special_tokens=special_tokens)
raw_zh_tokenizer.train(
  files=[TARGET_FILE],
  trainer=trainer
)

In [6]:
raw_en_tokenizer.encode("Hello, world!").tokens

['Hello', ',', 'world', '!']

In [7]:
raw_zh_tokenizer.encode("你好，世界").tokens

['你', '好，', '世界']

In [8]:
from transformers import PreTrainedTokenizerFast

en_tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw_en_tokenizer)
en_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

zh_tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw_zh_tokenizer)
zh_tokenizer.add_special_tokens({'pad_token': '[PAD]', 'bos_token': '[SOS]', 'eos_token': '[EOS]'})

# Padding base on max_length, it will keep adding <PAD> until it reaches max_length
print(en_tokenizer("You smell money", return_tensors="pt", padding="max_length", max_length=16))
print(zh_tokenizer("你闻到的是钱的气味", return_tensors="pt", padding="max_length", max_length=16))

{'input_ids': tensor([[ 360, 4015,  936,    3,    3,    3,    3,    3,    3,    3,    3,    3,
            3,    3,    3,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}
{'input_ids': tensor([[  339, 20821,  4779, 19522, 11325,     3,     3,     3,     3,     3,
             3,     3,     3,     3,     3,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}


# Transformer Model
Following is the simple implementation of the transformer model including encoder and decoder.

## Encoder

In [9]:
import torch
from torch import nn

class EncoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1
    ):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm((d_model, ))
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, X, src_padding_mask):
        """X has shape (batch_size, sequence_length, d_model)"""
        # Apply multi-head attention to get new representation
        attn_output, _ = self.attention(X, X, X, key_padding_mask=src_padding_mask)
        # Add & Norm for the multi-head attention output
        norm1_input = X + attn_output
        norm1_output = self.norm1(norm1_input)
        # Feed forward
        ffn_output = self.linear_relu_stack(norm1_output)
        ffn_output = self.dropout2(ffn_output)
        # Add & Norm for the FFN output
        norm2_input = norm1_output + ffn_output
        norm2_output = self.norm2(norm2_input)
        return norm2_output

## Decoder

In [10]:
class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        # The masked Multi-head attention
        self.masked_self_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        self.dropout1 = nn.Dropout(dropout)
        # The attention that came from the encoder
        self.encoder_decoder_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm((d_model, ))
        self.dropout2 = nn.Dropout(dropout)
        # The feed forward (FFN) part
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm3 = nn.LayerNorm(d_model) # Normalizes over the d_model dimension
        self.dropout3 = nn.Dropout(dropout) # Dropout after FFN residual

    def forward(
        self,
        target_input,
        encoder_output,
        src_padding_mask=None,
        trg_padding_mask=None,
        trg_attn_mask=None,
    ):
        """
        Decoder will need to take in the encoder output as part of the input
        """
        residual_input = target_input
        # --- Masked Multi-Head Self-Attention ---
        # Query, Key, Value are the same (from the decoder's path)
        # Pass the trg_attn_mask (causal), and trg_padding_mask
        masked_attn_output, _ = self.masked_self_attention(
            query=target_input,
            key=target_input,
            value=target_input,
            attn_mask=trg_attn_mask,
            key_padding_mask=trg_padding_mask,
        )

        # --- Add & Norm 1 ---
        # Add residual connection (input to this sub-layer)
        output_after_self_attn = residual_input + masked_attn_output
        # Apply dropout
        output_after_self_attn = self.dropout1(output_after_self_attn)
        # Apply Layer Norm
        norm1_output = self.norm1(output_after_self_attn) # Shape (batch_size, target_seq_len, d_model)

        # Store input for next residual connection
        residual_input = norm1_output # Input to encoder-decoder attention

        # --- Multi-Head Encoder-Decoder Attention (Cross-Attention) ---
        # Query from decoder's path (output of first norm)
        # Key and Value from encoder's output
        # src_padding_mask can be used here to mask padding in the encoder output
        cross_attn_output, _ = self.encoder_decoder_attention(
            query=norm1_output,
            key=encoder_output,
            value=encoder_output,
            key_padding_mask=src_padding_mask # Apply encoder padding mask here if needed
        )

        # --- Add & Norm 2 ---
        # Add residual connection (input to this sub-layer)
        output_after_cross_attn = residual_input + cross_attn_output
        # Apply dropout
        output_after_cross_attn = self.dropout2(output_after_cross_attn)
        # Apply Layer Norm
        norm2_output = self.norm2(output_after_cross_attn) # Shape (batch_size, target_seq_len, d_model)

        # Store input for next residual connection
        residual_input = norm2_output # Input to Feed-Forward Network

        # --- Feed-Forward Network ---
        # FFN operates independently on the last dimension
        ffn_output = self.linear_relu_stack(norm2_output) # Shape (batch_size, target_seq_len, d_model)
        # Apply dropout
        ffn_output = self.dropout3(ffn_output) # Dropout after FFN output

        # --- Add & Norm 3 ---
        # Add residual connection (input to this sub-layer)
        output_after_ffn = residual_input + ffn_output
        # Apply Layer Norm
        norm3_output = self.norm3(output_after_ffn) # Shape (batch_size, target_seq_len, d_model)

        return norm3_output # Output of the Decoder Block

## Encoder Decoder Combined

In [54]:
import math

class EncoderDecoderTransformer(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        vocab_size_source: int,
        vocab_size_target: int,
        num_encoder_layers: int,
        num_decoder_layers: int,
        max_seq_len: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_model = d_model
        # Embeddings
        self.source_embeddings = nn.Embedding(vocab_size_source, d_model)
        self.target_embeddings = nn.Embedding(vocab_size_target, d_model)
        # Positional encoding is fixed, therefore we register it in buffer
        pe = self._generate_fixed_positional_encoding(max_seq_len, d_model)
        self.register_buffer('positional_encoding', pe)
        # Encoder layers
        self.encoder_stack = nn.ModuleList([
            EncoderBlock(d_model, heads, d_ff, dropout) for _ in range(num_encoder_layers)
        ])
        # Decoder layers
        self.decoder_stack = nn.ModuleList([
            DecoderBlock(d_model, heads, d_ff, dropout) for _ in range(num_decoder_layers)
        ])
        # Output layer
        self.output_layer = nn.Linear(d_model, vocab_size_target)
        # Initialize weights
        self._initialize_parameters()

    def _initialize_parameters(self):
        # Common initialization for Transformer weights
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _generate_fixed_positional_encoding(self, max_seq_len: int, d_model: int):
        """Generate a matrix for fixed positional encoding base on the max_seq_len"""
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Add batch dimension (1, max_seq_len, d_model)
        return pe # This will be added to embeddings

    def encode(self, src_tokens, src_padding_mask=None):
      src_embed = self.source_embeddings(src_tokens) * math.sqrt(self.d_model) # Scale embeddings
      src_embed = src_embed + self.positional_encoding[:, :src_tokens.size(1), :]
      src_embed = nn.Dropout(self.dropout)(src_embed)
      encoder_output = src_embed
      for encoder_layer in self.encoder_stack:
        encoder_output = encoder_layer(encoder_output, src_padding_mask=src_padding_mask)
      return encoder_output

    def decode(
        self,
        trg_tokens,
        encoder_output,
        src_padding_mask=None,
        trg_padding_mask=None,
        trg_attn_mask=None,
    ):
      trg_embed = self.target_embeddings(trg_tokens) * math.sqrt(self.d_model) # Scale embeddings
      trg_embed = trg_embed + self.positional_encoding[:, :trg_tokens.size(1), :]
      trg_embed = nn.Dropout(self.dropout)(trg_embed)
      decoder_output = trg_embed
      for decoder_layer in self.decoder_stack:
        decoder_output = decoder_layer(
            decoder_output,
            encoder_output,
            src_padding_mask=src_padding_mask,
            trg_padding_mask=trg_padding_mask,
            trg_attn_mask=trg_attn_mask,
        )
      output_logits = self.output_layer(decoder_output)
      return output_logits

    def forward(
        self,
        src_tokens,
        trg_tokens,
        trg_padding_mask=None,
        src_padding_mask=None
    ):
        """
        This is a high-level forward pass outline. Actual implementation needs mask handling.
        Masks need to be generated based on src_tokens and trg_tokens padding
        Lookahead mask for decoder self-attention also needs to be generated.
        """
        # Encoder Pass
        encoder_output = self.encode(src_tokens, src_padding_mask=src_padding_mask)

        # Decoder Pass
        # Generate causal mask for decoder self-attention (shape seq_len, seq_len)
        causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(
            trg_tokens.size(1),
            device=trg_tokens.device
        )
        return self.decode(
            trg_tokens,
            encoder_output,
            src_padding_mask=src_padding_mask,
            trg_padding_mask=trg_padding_mask,
            trg_attn_mask=causal_mask,
        )

# Prepare Training Data
Next we will prepare some training data. We will use the same one that we used for the tokenizers.

In [55]:
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, src_path: str, tgt_path: str):
        with open(src_path, 'r', encoding='utf-8') as f:
            self.src_lines = [line.strip() for line in f]
        with open(tgt_path, 'r', encoding='utf-8') as f:
            self.tgt_lines = [line.strip() for line in f]
        assert len(self.src_lines) == len(self.tgt_lines), "Mismatch in number of lines"

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        src = self.src_lines[idx]
        tgt = self.tgt_lines[idx]
        return (src, tgt)

In [56]:
dataset = TranslationDataset(
    src_path=MINI_SOURCE_FILE,
    tgt_path=MINI_TARGET_FILE,
)
dataset[0:5]

(['http://www.ted.com/talks/stephen_palumbi_following_the_mercury_trail.html',
  "There's a tight and surprising link between the ocean's health and ours, says marine biologist Stephen Palumbi. He shows how toxins at the bottom of the ocean food chain find their way into our bodies, with a shocking story of toxic contamination from a Japanese fish market. His work points a way forward for saving the oceans' health -- and humanity's.",
  'fish,health,mission blue,oceans,science',
  '899',
  'Stephen Palumbi: Following the mercury trail'],
 ['http://www.ted.com/talks/lang/zh-cn/stephen_palumbi_following_the_mercury_trail.html',
  '生物学家史蒂芬·帕伦认为，海洋的健康和我们的健康之间有着紧密而神奇的联系。他通过日本一个渔场发生的让人震惊的有毒污染的事件，展示了位于海洋食物链底部的有毒物质是如何进入我们的身体的。他的工作主要是未来拯救海洋健康的方法——同时也包括人类的。',
  'fish,health,mission blue,oceans,science',
  '899',
  '史蒂芬·帕伦：追寻水银的踪迹'])

Then we will split the data into training and testing set, create batches, and shuffling them.

In [57]:
from torch.utils.data import DataLoader, random_split

train_data, test_data = random_split(dataset, [0.8, 0.2])
print("Training data size:", len(train_data), ", Test data size:", len(test_data))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

sample_train_input, sample_train_output = next(iter(train_loader))
print(sample_train_input[0])
print(sample_train_output[0])

Training data size: 80 , Test data size: 20
This is whale meat that I photographed in a grocery store in Tokyo -- or is it?
这是鲸鱼肉 我在东京一家食品店拍到的 或者不是？


# Training
Finally, in the actual training, we will implment a `train_one_epoch` function that do one round of training over the training data. Then we will train it a few epochs to see how it goes.

In [58]:
def get_padding_mask(input_ids, pad_token_id, device):
      return torch.where(
          input_ids.to(device) == pad_token_id,
          torch.tensor(float('-inf'), device=device),
          torch.tensor(0.0, device=device),
      )

In [59]:
import torch

def train_one_epoch(transformer, train_loader, en_tokenizer, zh_tokenizer, optimizer, criterion, device, max_seq_len, limit_batches=None):
    running_loss = 0.0
    total_batches = len(train_loader)
    processed_batches = 0

    pad_token_id = zh_tokenizer.pad_token_id
    sos_token_id = zh_tokenizer.bos_token_id

    if pad_token_id is None or sos_token_id is None:
         raise ValueError("Target tokenizer must have pad_token_id and bos_token_id defined.")

    transformer.train() # Set model to training mode
    for input_seq, output_seq in train_loader:
        optimizer.zero_grad()

        # Tokenization
        # input_seq shape (batch_size, max_seq_len)
        # output_seq shape (batch_size, max_seq_len)
        en_tokens = en_tokenizer(
            input_seq, return_tensors="pt", padding="max_length", truncation=True, max_length=max_seq_len
        )
        zh_tokens = zh_tokenizer(
            output_seq, return_tensors="pt", padding="max_length", truncation=True, max_length=max_seq_len
        )
        en_input_ids = en_tokens["input_ids"].to(device)
        zh_input_ids = zh_tokens["input_ids"].to(device)

        # Prepare padding mask
        encoder_padding_mask = get_padding_mask(
            en_input_ids, en_tokenizer.pad_token_id, device
        )
        decoder_padding_mask = get_padding_mask(
            zh_input_ids, zh_tokenizer.pad_token_id, device
        )

        # Prepare Decoder Input (shifted right: <SOS> + target tokens[:-1])
        batch_size = zh_input_ids.size(0)
        decoder_input_ids = torch.full(
            (batch_size, 1), sos_token_id, device=device, dtype=zh_input_ids.dtype
        )
        decoder_input_ids = torch.cat(
            [decoder_input_ids, zh_input_ids[:, :-1]], dim=1
        )

        # Target for loss is the original Chinese input IDs
        target_tokens_for_loss = zh_input_ids

        # Forward pass
        # The main model's forward method is assumed to accept these additive masks
        # and handle causal mask generation internally, combining it with decoder_padding_mask for decoder self-attention
        # and passing encoder_padding_mask to decoder cross-attention.
        predictions = transformer(
            src_tokens=en_input_ids,
            trg_tokens=decoder_input_ids, # Pass the shifted decoder input
            src_padding_mask=encoder_padding_mask, # Additive mask for encoder self-attention
            trg_padding_mask=decoder_padding_mask, # Additive mask for decoder self-attention (combined with causal) and cross-attention
        )

        # Calculate Loss
        # CrossEntropyLoss expects input (N, C) and target (N)
        # It will ignore the pad_token_id in target_tokens_for_loss
        loss = criterion(
            predictions.view(-1, predictions.size(-1)),
            target_tokens_for_loss.view(-1)
        )

        # Backpropagation and Optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        processed_batches += 1

        # Check if limit is reached
        if limit_batches is not None and processed_batches >= limit_batches:
            break

    # Calculate average loss based on processed batches
    avg_loss = running_loss / processed_batches if processed_batches > 0 else 0.0
    return avg_loss

We have mini files that contain only 100 lines of code. We will use them to verify that our code works correctly. Since it is very small, it should train and overfit very fast.

In [60]:
import torch
from torch import nn
import torch.optim as optim

num_epochs = 100
max_seq_len = 16
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
transformer_model = EncoderDecoderTransformer(
    d_model=64,
    heads=2,
    vocab_size_source=len(en_tokenizer),
    vocab_size_target=len(zh_tokenizer),
    num_encoder_layers=2,
    num_decoder_layers=2,
    max_seq_len=max_seq_len,
    d_ff=256,
    dropout=0.1
)
optimizer = optim.Adam(transformer_model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=zh_tokenizer.pad_token_id) # Ignore padding in loss
transformer_model.to(device) # Move model to device

# Keep training until the loss is less than 2
loss_limit = 0.01
current_epoch = 0
train_loss = float('inf')
max_epoch = 800
report_every = 50
while train_loss > loss_limit:
    # Train for one epoch
    train_loss = train_one_epoch(
        transformer_model,
        train_loader,
        en_tokenizer,
        zh_tokenizer,
        optimizer,
        criterion,
        device,
        max_seq_len,
    )
    current_epoch += 1
    if current_epoch % report_every == 0:
        print(f"Epoch {current_epoch}/{max_epoch}, Loss: {train_loss:.4f}")

    if current_epoch > max_epoch:
        print(f"Did not reach {loss_limit} after {max_epoch} epochs. Loss: {train_loss:.4f}. Training stopped.")
        break

print(f"Final loss: {train_loss:.4f}")
print("Training finished.")

Epoch 50/800, Loss: 5.5389
Epoch 100/800, Loss: 2.1362
Epoch 150/800, Loss: 0.4631
Epoch 200/800, Loss: 0.1546
Epoch 250/800, Loss: 0.0793
Epoch 300/800, Loss: 0.0490
Epoch 350/800, Loss: 0.0322
Epoch 400/800, Loss: 0.0249
Epoch 450/800, Loss: 0.0198
Epoch 500/800, Loss: 0.0146
Epoch 550/800, Loss: 0.0132
Final loss: 0.0099
Training finished.


Observe that the loss is gradually decreasing, this is a good sign that the model is learning from the training set.

In [64]:
def predict(transformer, input_seq, en_tokenizer, zh_tokenizer, device, max_seq_len):
    transformer.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        # Tokenize input
        input_tokens = en_tokenizer(
            input_seq, return_tensors="pt", padding="max_length", truncation=True, max_length=max_seq_len
        )

        # Prepare padding mask
        # Shape (1, max_seq_len)
        input_token_ids = input_tokens["input_ids"].to(device)
        input_padding_mask = get_padding_mask(
          input_token_ids,
          en_tokenizer.pad_token_id,
          device
        )

        # Get encoder output
        # Shape (1, max_seq_len, d_model)
        encoder_output = transformer.encode(
            input_token_ids, src_padding_mask=input_padding_mask
        )

        # Initialize decoder input
        sos_token_id = zh_tokenizer.bos_token_id
        pad_token_id = zh_tokenizer.pad_token_id
        eos_token_id = zh_tokenizer.eos_token_id

        # Start by a <SOS> token in the output
        # Shape (1, 1)
        decoder_input_ids = torch.tensor([[sos_token_id]], device=device)

        for _ in range(max_seq_len):
            current_seq_len = decoder_input_ids.size(1)

            # --- Prepare Decoder Input Tensor for Model ---
            # Pad current decoder input sequence to max_seq_len
            # Fill the padded_decoder_input_ids with <PAD>
            # Shape: (1, max_seq_len)
            padded_decoder_input_ids = torch.full(
                (decoder_input_ids.size(0), max_seq_len),
                pad_token_id,
                device=device,
                dtype=decoder_input_ids.dtype
            )
            # Place whatever we have to the padded_decoder_input_ids
            # Shape: (1 ,max_seq_len)
            padded_decoder_input_ids[:, :current_seq_len] = decoder_input_ids

            # Prepare padding mask
            # Shape: (1, max_seq_len)
            decoder_padding_mask = get_padding_mask(
                padded_decoder_input_ids,
                zh_tokenizer.pad_token_id,
                device
            )

            # Prepare the attention mask
            # Shape: (max_seq_len, max_seq_len)
            # It is a triangular matrix where lower triangle is 0 and upper
            # triangle is float('-inf').
            causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(
                max_seq_len,
                device=padded_decoder_input_ids.device
            )

            # Get decoder output logits
            # Shape: (1, max_seq_len, vocab_size)
            decoder_output_logits = transformer.decode(
                padded_decoder_input_ids,
                encoder_output,
                src_padding_mask=input_padding_mask,
                trg_padding_mask=decoder_padding_mask,
                trg_attn_mask=causal_mask,
            )

            # Get the logit of the next token
            # The shape of the decoder output is (1, max_seq_len, vocab_size)
            # We will select the logit at the current_seq_len - 1 because this
            # is the token that we are predicting.
            # Shape: (1, vocab_size)
            next_token_logits = decoder_output_logits[:, current_seq_len - 1, :]
            # Get predicted token ID (Greedy decoding)
            # Apply softmax on the last dimension of next_token_logits, which has
            # size vocab_size.
            # Shape: (1, vocab_size)
            next_token_probs = torch.softmax(next_token_logits, dim=-1)

            # Use argmax to select the token id with the highest probability
            predicted_next_token_id = torch.argmax(next_token_probs, dim=-1).item()

            # Append the predicted token to the sequence for the next iteration
            # Shape (1, current_seq_len + 1)
            decoder_input_ids = torch.cat(
                [decoder_input_ids, torch.tensor([[predicted_next_token_id]], device=device)],
                dim=1
            )

            if predicted_next_token_id == eos_token_id:
                break # Stop if EOS token is predicted
            if current_seq_len + 1 >= max_seq_len:
                break # Stop if max length is reached

        # Convert batch into one single list
        # Shape (max_seq_len,)
        predicted_ids_list = decoder_input_ids[0].tolist()
        if eos_token_id in predicted_ids_list:
            eos_index = predicted_ids_list.index(eos_token_id)
        else:
            eos_index = len(predicted_ids_list)

        # Exclude SOS token (first token) and EOS token (if present)
        translation_ids = predicted_ids_list[1:eos_index]

        # Convert token IDs back to a string
        translation = zh_tokenizer.decode(translation_ids, skip_special_tokens=True)

        return translation


In [90]:
train_input_data, train_output_data = next(iter(train_loader))
print(train_input_data[0])
print(train_output_data[0])
predict(transformer_model, train_input_data[0], en_tokenizer, zh_tokenizer, device, max_seq_len)

And it can be a very complicated thing, what human health is.
人类的健康也是一件非常复杂的事情。


'人类的 健康 也是 一件 非常复杂 的事情 。 人类的 健康 也是 一件 非常复杂 的事情 。 人类的'

In [95]:
test_input_data, test_output_data = next(iter(test_loader))
print(test_input_data[0])
print(test_output_data[0])
predict(transformer_model, test_input_data[0], en_tokenizer, zh_tokenizer, device, max_seq_len)

It had two-to-three-to-400 times the toxic loads ever allowed by the EPA.
含有的毒素是有史以来环保局允许的 2-3倍到400倍。


'这 导致了 日本 一系列的 其他 运动 。 在 这一点 上，我 真的 非常 骄傲 的说 ， 在 日本'

Base on the above examples we can observe a few things:
1. The model is predicting well on the training set, meaning that our model implementation, training loop, and predict function is likely correct.
2. The model does not know where to stop, it will only stop when the `max_seq_len` is reached. This is due to in our training set, we don't have that special `[EOS]` token. A solution could be adding them for every line in the training file.
3. The model is doing bad on the testing set, which is understandable because:
  1. The training size is too small and the model is overfitted
  2. The Chinese tokenizer is not doing very well

# Summary
In summary, we successfully implemented a Encoder-Decoder Transformer, trained it on a small set of training data, and implemented the predict function correctly. There are many things that we can improve:
1. Make a validation set for hyperparameters tunning
2. Train the model on the actual training set instead of the mini one (require high computation resources)